In [31]:
from co2sat.data.static import (
    build_epa_statics,
    extract_edgar_at_plants,
    extract_raster_at_plants,
)
from co2sat.utils import data_dir
import plotly.express as px
import xarray as xr
import numpy as np
import pandas as pd
import rasterio

In [32]:
statics = build_epa_statics(data_dir("processed", "epa_daily_with_attributes.parquet"))
statics.head()

,facility_id,latitude,longitude,capacity_mw,coal_ratio,gas_ratio,oil_ratio,other_ratio,zenith_angle
0,3,31.0069,-88.0103,2946.4,0.267717,0.595235,0.0,0.137049,38.728350
1,10,32.6017,-87.7811,1288.5,0.000000,1.000000,0.0,0.000000,40.302477
2,26,33.2442,-86.4567,2012.8,0.472973,0.527027,0.0,0.000000,40.524548
3,47,34.7439,-87.8486,476.0,0.000000,1.000000,0.0,0.000000,42.567788
4,51,32.0306,-93.5692,720.8,1.000000,0.000000,0.0,0.000000,42.264308


In [33]:
print(statics.shape)  # expect (1158, 9)
statics.describe().T

(1158, 9)


,count,mean,std,min,25%,50%,75%,max
facility_id,1158.0,28191.773748,25840.644769,3.0,3297.25,8062.5,55400.75,70454.0
latitude,1158.0,37.443839,4.922194,24.5636,33.40925,38.39455,41.0703,48.9905
longitude,1158.0,-91.483625,14.284041,-123.48,-97.73455,-88.9839,-80.624125,-68.7106
capacity_mw,1158.0,765.751822,769.798503,21.3,189.0,540.75,1061.6,7425.0
coal_ratio,1158.0,0.163974,0.360615,0.0,0.0,0.0,0.0,1.0
gas_ratio,1158.0,0.795296,0.389657,0.0,1.0,1.0,1.0,1.0
oil_ratio,1158.0,0.037474,0.179209,0.0,0.0,0.0,0.0,1.0
other_ratio,1158.0,0.003256,0.044335,0.0,0.0,0.0,0.0,1.0
zenith_angle,1158.0,48.59311,7.696062,29.653002,43.69984,47.374554,52.113218,71.839462


In [34]:
fig = px.scatter(
    statics,
    x="longitude",
    y="latitude",
    color="zenith_angle",
    color_continuous_scale="Viridis",
    height=500,
    width=900,
    title="Satellite zenith angle across the fleet — expect smooth SE→NW gradient",
    hover_data=["facility_id", "capacity_mw"],
)
fig.show()

In [35]:
statics.to_parquet(data_dir("processed", "static_features.parquet"), index=False)
print("saved:", data_dir("processed", "static_features.parquet"))

saved: /home/karim/co2-satellite-replication/data/processed/static_features.parquet


In [36]:
nc_path = next((data_dir("raw", "edgar")).glob("*.nc"))
ds = xr.open_dataset(nc_path)
print(ds)

<xarray.Dataset> Size: 26MB
Dimensions:    (lat: 1800, lon: 3600)
Coordinates:
  * lat        (lat) float64 14kB -89.95 -89.85 -89.75 ... 89.75 89.85 89.95
  * lon        (lon) float64 29kB -179.9 -179.8 -179.8 ... 179.8 179.8 179.9
Data variables:
    emissions  (lat, lon) float32 26MB ...
Attributes:
    institution:       European Commission, Joint Research Centre
    source:            https://edgar.jrc.ec.europa.eu/dataset_ghg80
    how_to_cite:       https://edgar.jrc.ec.europa.eu/dataset_ghg80#howtocite
    copyright_notice:  https://edgar.jrc.ec.europa.eu/dataset_ghg80#conditions
    contacts:          https://edgar.jrc.ec.europa.eu/dataset_ghg80#info JRC-...


In [37]:
var = list(ds.data_vars)[0]
print("\nVariable:", var)
print("Units:", ds[var].attrs.get("units"))
print("Dims:", ds[var].dims, ds[var].shape)

print("\nLon range:", float(ds["lon"].min()), "→", float(ds["lon"].max()))
print("First lons:", ds["lon"].values[:4])
print("First lats:", ds["lat"].values[:4])


Variable: emissions
Units: Tonnes
Dims: ('lat', 'lon') (1800, 3600)

Lon range: -179.95 → 179.95
First lons: [-179.95 -179.85 -179.75 -179.65]
First lats: [-89.95 -89.85 -89.75 -89.65]


In [38]:
statics = pd.read_parquet(data_dir("processed", "static_features.parquet"))
statics["edgar_co2_surround"] = extract_edgar_at_plants(nc_path, statics)

In [39]:
# 1. Positivity — a zero/negative would mean an ocean/empty cell (bad coords or offset bug)
print("Non-positive:", (statics["edgar_co2_surround"] <= 0).sum())

Non-positive: 0


In [40]:
# 2. Distribution — log-normal-ish
px.histogram(
    np.log10(statics["edgar_co2_surround"]),
    nbins=60,
    title="log10 EDGAR CO2 of containing cell",
).show()

In [41]:
# 3. Geography — hotspots must be cities/industrial corridors
px.scatter(
    statics,
    x="longitude",
    y="latitude",
    color=np.log10(statics["edgar_co2_surround"]),
    color_continuous_scale="Inferno",
    height=500,
    width=900,
    hover_data=["facility_id"],
    title="log10 EDGAR surroundings — expect urban/industrial hotspots",
).show()

In [42]:
shifts = [-0.1, 0.0, 0.1]
neighbor_vals = []

for dlat in shifts:
    for dlon in shifts:
        shifted = statics.copy()
        shifted["latitude"] = statics["latitude"] + dlat
        shifted["longitude"] = statics["longitude"] + dlon
        neighbor_vals.append(extract_edgar_at_plants(nc_path, shifted))

mean_3x3 = np.mean(neighbor_vals, axis=0)
single = statics["edgar_co2_surround"].values

In [43]:
r = np.corrcoef(np.log10(single), np.log10(mean_3x3))[0, 1]
print(f"containing-cell vs 3x3-mean correlation (log10): r = {r:.3f}")

containing-cell vs 3x3-mean correlation (log10): r = 0.782


Variants correlate at r = 0.78; the definition carries real sensitivity. We use the containing cell (most literal reading) since the original paper did not specify whether they took containing cell for each plant when considering "surrounding carbon emissions" or a neighborhood window.

In [44]:
tif_path = next(data_dir("raw", "consumption").glob("*.tif"))
print(tif_path.name)
with rasterio.open(tif_path) as src:
    print("CRS:", src.crs)
    print("Size:", src.width, "x", src.height)
    print("Resolution:", src.res)
    print("Bounds:", src.bounds)
    print("Nodata:", src.nodata)
    print("Dtype:", src.dtypes)

EC2019.tif
CRS: ESRI:54009
Size: 36080 x 15517
Resolution: (1000.0, 1000.0)
Bounds: BoundingBox(left=-18040094.547525823, bottom=-7343860.161923394, right=18039905.452474177, top=8173139.838076606)
Nodata: -3.4028230607370965e+38
Dtype: ('float32',)


In [45]:
statics = pd.read_parquet(data_dir("processed", "static_features.parquet"))
statics["consumption_surround"] = extract_raster_at_plants(tif_path, statics)

print("NaN:", statics["consumption_surround"].isna().sum())
print("Zeros:", (statics["consumption_surround"] == 0).sum())
print(statics["consumption_surround"].describe())

NaN: 57
Zeros: 0
count    1.101000e+03
mean     4.638103e+06
std      3.325483e+06
min      6.457348e+04
25%      1.479554e+06
50%      4.048426e+06
75%      8.145863e+06
max      1.201351e+07
Name: consumption_surround, dtype: float64


In [46]:
# EDGAR feature — was computed but not persisted
nc_path = next(data_dir("raw", "edgar").glob("*.nc"))
statics["edgar_co2_surround"] = extract_edgar_at_plants(nc_path, statics)

In [47]:
vals = statics["consumption_surround"]

# Dark cells: fill NaN and keep zeros as 0 (nightlight-dark = negligible local consumption)
statics["consumption_surround"] = vals.fillna(0)

# Distribution (log, +1 for zeros)
px.histogram(
    np.log10(statics["consumption_surround"] + 1),
    nbins=60,
    title="log10(EC2019 + 1) at plant cells",
).show()

# Geography — should literally be city lights sampled at plants
px.scatter(
    statics,
    x="longitude",
    y="latitude",
    color=np.log10(statics["consumption_surround"] + 1),
    color_continuous_scale="Cividis",
    height=500,
    width=900,
    title="Consumption surroundings — expect metro-area hotspots",
).show()

# Cross-check vs EDGAR (both are neighborhood-activity proxies)
mask = (statics["consumption_surround"] > 0) & (statics["edgar_co2_surround"] > 0)
r = np.corrcoef(
    np.log10(statics.loc[mask, "consumption_surround"]),
    np.log10(statics.loc[mask, "edgar_co2_surround"]),
)[0, 1]
print(f"log-log correlation consumption vs EDGAR: r = {r:.3f}")

log-log correlation consumption vs EDGAR: r = 0.218


In [48]:
# Testing the transformation of ESRI I did before
probe = pd.DataFrame(
    {
        "name": [
            "Downtown Houston",
            "Manhattan",
            "Chicago Loop",
            "Rural Nevada",
            "West Texas desert",
            "Montana plains",
        ],
        "latitude": [29.7604, 40.7549, 41.8837, 39.50, 31.20, 47.00],
        "longitude": [-95.3698, -73.9840, -87.6289, -116.50, -103.50, -108.50],
    }
)
probe["ec"] = extract_raster_at_plants(tif_path, probe)
print(probe)

                name  latitude  longitude            ec
0   Downtown Houston   29.7604   -95.3698  9.395758e+06
1          Manhattan   40.7549   -73.9840  9.395758e+06
2       Chicago Loop   41.8837   -87.6289  9.395758e+06
3       Rural Nevada   39.5000  -116.5000  3.654389e+04
4  West Texas desert   31.2000  -103.5000  1.416445e+06
5     Montana plains   47.0000  -108.5000  3.654389e+04


In [49]:
facilities = pd.read_parquet(
    data_dir("processed", "epa_daily_with_attributes.parquet")
)[["facility_id", "facility_name", "state"]].drop_duplicates("facility_id")
ranked = statics.merge(facilities, on="facility_id")[
    ["facility_name", "state", "latitude", "longitude", "consumption_surround"]
].sort_values("consumption_surround")
print("=== BOTTOM 8 (expect rural/remote) ===")
print(ranked.head(8).to_string(index=False))
print("=== TOP 8 (expect urban/metro) ===")
print(ranked.tail(8).to_string(index=False))

=== BOTTOM 8 (expect rural/remote) ===
                 facility_name state  latitude  longitude  consumption_surround
   Riviera Beach Energy Center    FL   26.7653   -80.0528                   0.0
Lansing Smith Generating Plant    FL   30.2689   -85.7003                   0.0
                  Turkey Point    FL   25.4356   -80.3308                   0.0
                    P L Bartow    FL   27.8613   -82.6012                   0.0
                Cape Canaveral    FL   28.4694   -80.7642                   0.0
               Port Everglades    FL   26.0856   -80.1253                   0.0
              Newark Bay Cogen    NJ   40.7197     -74.13                   0.0
          Newark Energy Center    NJ   40.7082   -74.1284                   0.0
=== TOP 8 (expect urban/metro) ===
                 facility_name state  latitude  longitude  consumption_surround
                      Cherokee    CO   39.8078  -104.9648            10176709.0
    LaPorte Generating Station    TX    29.702

In [50]:
statics.to_parquet(data_dir("processed", "static_features.parquet"), index=False)
print(statics.shape)  # expect (1158, 12): facility_id + 11 features
print(statics.columns.tolist())

(1158, 11)
['facility_id', 'latitude', 'longitude', 'capacity_mw', 'coal_ratio', 'gas_ratio', 'oil_ratio', 'other_ratio', 'zenith_angle', 'consumption_surround', 'edgar_co2_surround']


In [51]:
FEATURES = [
    "capacity_mw",
    "latitude",
    "longitude",
    "coal_ratio",
    "gas_ratio",
    "oil_ratio",
    "other_ratio",
    "altitude_m",
    "zenith_angle",
    "edgar_co2_surround",
    "consumption_surround",
]
assert list(statics.columns) == ["facility_id"] + FEATURES or set(FEATURES) <= set(
    statics.columns
)
print(statics[FEATURES].describe().T.round(2))
print("\nNaN per column:\n", statics[FEATURES].isna().sum())

AssertionError: 

In [53]:
FEATURES = [
    "capacity_mw",
    "latitude",
    "longitude",
    "coal_ratio",
    "gas_ratio",
    "oil_ratio",
    "other_ratio",
    "altitude_m",
    "zenith_angle",
    "edgar_co2_surround",
    "consumption_surround",
]
assert list(statics.columns) == ["facility_id"] + FEATURES or set(FEATURES) <= set(
    statics.columns
)
print(statics[FEATURES].describe().T.round(2))
print("\nNaN per column:\n", statics[FEATURES].isna().sum())

AssertionError: 

In [54]:
print(statics.columns.tolist())
print(set(FEATURES) - set(statics.columns))  # names I used that you don't have

['facility_id', 'latitude', 'longitude', 'capacity_mw', 'coal_ratio', 'gas_ratio', 'oil_ratio', 'other_ratio', 'zenith_angle', 'consumption_surround', 'edgar_co2_surround']
{'altitude_m'}


In [55]:
cache_path = data_dir("interim", "altitude_cache.parquet")
print("cache exists:", cache_path.exists())
if cache_path.exists():
    alt = pd.read_parquet(cache_path)
    print(alt.shape, "| NaN:", alt["altitude_m"].isna().sum())
    print(alt["altitude_m"].describe())

cache exists: True
(1158, 2) | NaN: 0
count    1158.000000
mean      256.915877
std       375.365229
min       -28.087317
25%        25.091878
50%       149.222855
75%       265.826691
max      2126.706299
Name: altitude_m, dtype: float64


In [56]:
statics = pd.read_parquet(data_dir("processed", "static_features.parquet"))
statics = statics.merge(alt, on="facility_id", how="left")
print("missing altitude:", statics["altitude_m"].isna().sum())  # want 0
statics.to_parquet(data_dir("processed", "static_features.parquet"), index=False)

# The guard — reload and assert, every save from now on
check = pd.read_parquet(data_dir("processed", "static_features.parquet"))
assert "altitude_m" in check.columns and check["altitude_m"].notna().all()
print("persisted ✓", check.shape)

missing altitude: 0
persisted ✓ (1158, 12)
